# Competitive Coevolution: Predator vs Prey

In this notebook, we will implement a simple competitive coevolution system.

We will evolve two populations:
- Predators (try to catch prey)
- Prey (try to avoid predators)

Key idea:
Fitness depends on interactions between individuals, not an absolute objective.

## Key Concepts

- Each individual’s fitness depends on its opponents
- Populations evolve simultaneously
- This creates a moving fitness landscape
- Often leads to:
  - Arms races
  - Cycling behavior
  - Red Queen dynamics

We will simulate this in a simplified 1D environment.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

np.random.seed(0)
random.seed(0)

## Simplified Environment

We represent each individual as a single real number.

Interpretation:
- Predator and prey positions on a 1D line
- Distance between them determines interaction

Fitness:
- Predator: wants to minimize distance
- Prey: wants to maximize distance

In [ ]:
# population initialization
pop_size = 30
generations = 50

def init_population():
    return np.random.uniform(-1, 1, pop_size)

predators = init_population()
prey = init_population()

## Fitness Function

We compute fitness based on pairwise interactions:

- Predator fitness increases when it is close to prey
- Prey fitness increases when it is far from predators

We approximate this by sampling interactions between individuals.

In [ ]:
# evaluation function
def evaluate(predators, prey):
    predator_fitness = np.zeros(len(predators))
    prey_fitness = np.zeros(len(prey))

    for i, p in enumerate(predators):
        for q in prey:
            distance = abs(p - q)
            predator_fitness[i] += -distance  # minimize distance

    for j, q in enumerate(prey):
        for p in predators:
            distance = abs(p - q)
            prey_fitness[j] += distance  # maximize distance

    return predator_fitness, prey_fitness

## Evolution Operators

We use:
- Selection: keep the best half of each population
- Mutation: add Gaussian noise

No crossover is used here for simplicity.

In [ ]:
# selection and mutation
def select_parents(pop, fitness, num_parents):
    idx = np.argsort(fitness)[-num_parents:]
    return pop[idx]

def reproduce(parents, pop_size, mutation_std=0.05):
    children = []
    while len(children) < pop_size:
        parent = random.choice(parents)
        child = parent + np.random.normal(0, mutation_std)
        children.append(child)
    return np.array(children)

## Evolution Loop

Both populations evolve simultaneously:
1. Evaluate fitness
2. Select the best individuals
3. Apply mutation
4. Repeat

Observe how each population influences the other over time.

In [ ]:
predators = init_population()
prey = init_population()

history_pred = []
history_prey = []

for gen in range(generations):
    pred_fit, prey_fit = evaluate(predators, prey)

    history_pred.append(predators.copy())
    history_prey.append(prey.copy())

    # Selection (keep top half as parents)
    pred_parents = select_parents(predators, pred_fit, pop_size // 2)
    prey_parents = select_parents(prey, prey_fit, pop_size // 2)

    # Reproduction back to full population
    predators = reproduce(pred_parents, pop_size)
    prey = reproduce(prey_parents, pop_size)

    print(f"Gen {gen}: "
          f"Pred fitness={pred_fit.mean():.3f}, "
          f"Prey fitness={prey_fit.mean():.3f}")

## Observing the Dynamics

We now visualize how populations evolve over generations.

This helps us see:
- Movement of populations
- Clustering
- Possible oscillations

In [ ]:
plt.figure()

# for gen in range(0, 10, 1):
for gen in range(0, generations, 5):
    plt.scatter(history_pred[gen], [gen]*len(history_pred[gen]), label="Pred" if gen == 0 else "")
    plt.scatter(history_prey[gen], [gen]*len(history_prey[gen]), marker='x', label="Prey" if gen == 0 else "")

plt.xlabel("Strategy (1D position)")
plt.ylabel("Generation")
plt.title("Coevolution Dynamics")
plt.legend()
plt.show()

## Exercise 1 - Tournament Sampling
Modify the fitness evaluation so that each individual competes against a random subset of the opposite population instead of the full population.

Specifically:
- For each predator, sample k prey
- For each prey, sample k predators

Compare results with the original full evaluation.

Solutions here: https://github.com/giorgia-nadizar/evolution/blob/master/coevolution/solutions.py

In [ ]:
# evaluation function
def evaluate_tournament(predators, prey):
    predator_fitness = np.zeros(len(predators))
    prey_fitness = np.zeros(len(prey))

    # fill in here

    return predator_fitness, prey_fitness

In [ ]:
predators = init_population()
prey = init_population()

history_pred = []
history_prey = []

for gen in range(generations):
    pred_fit, prey_fit = evaluate_tournament(predators, prey)

    history_pred.append(predators.copy())
    history_prey.append(prey.copy())

    # Selection (keep top half as parents)
    pred_parents = select_parents(predators, pred_fit, pop_size // 2)
    prey_parents = select_parents(prey, prey_fit, pop_size // 2)

    # Reproduction back to full population
    predators = reproduce(pred_parents, pop_size)
    prey = reproduce(prey_parents, pop_size)

    print(f"Gen {gen}: "
          f"Pred fitness={pred_fit.mean():.3f}, "
          f"Prey fitness={prey_fit.mean():.3f}")

In [ ]:
plt.figure()

# for gen in range(0, 10, 1):
for gen in range(0, generations, 5):
    plt.scatter(history_pred[gen], [gen]*len(history_pred[gen]), label="Pred" if gen == 0 else "")
    plt.scatter(history_prey[gen], [gen]*len(history_prey[gen]), marker='x', label="Prey" if gen == 0 else "")

plt.xlabel("Strategy (1D position)")
plt.ylabel("Generation")
plt.title("Coevolution Dynamics")
plt.legend()
plt.show()

## Exercise 2 - Add a Hall of Fame
Maintain a Hall of Fame: store the best individuals from past generations.

When evaluating fitness:
- Include competition against Hall of Fame individuals

Solutions here: https://github.com/giorgia-nadizar/evolution/blob/master/coevolution/solutions.py

In [ ]:
hall_of_fame_pred = []
hall_of_fame_prey = []

def update_hof(pop, fitness, hof, top_k=3):
    idx = np.argsort(fitness)[-top_k:]
    hof.extend(pop[idx])
    return hof

def evaluate_with_hof(predators, prey, hof_pred, hof_prey):
    predator_fitness = np.zeros(len(predators))
    prey_fitness = np.zeros(len(prey))

    # fill in here

    return predator_fitness, prey_fitness

In [ ]:
predators = init_population()
prey = init_population()

history_pred = []
history_prey = []

for gen in range(generations):
    pred_fit, prey_fit = evaluate_with_hof(predators, prey)

    history_pred.append(predators.copy())
    history_prey.append(prey.copy())

    # Selection (keep top half as parents)
    pred_parents = select_parents(predators, pred_fit, pop_size // 2)
    prey_parents = select_parents(prey, prey_fit, pop_size // 2)

    # Reproduction back to full population
    predators = reproduce(pred_parents, pop_size)
    prey = reproduce(prey_parents, pop_size)

    print(f"Gen {gen}: "
          f"Pred fitness={pred_fit.mean():.3f}, "
          f"Prey fitness={prey_fit.mean():.3f}")

In [ ]:
plt.figure()

# for gen in range(0, 10, 1):
for gen in range(0, generations, 5):
    plt.scatter(history_pred[gen], [gen]*len(history_pred[gen]), label="Pred" if gen == 0 else "")
    plt.scatter(history_prey[gen], [gen]*len(history_prey[gen]), marker='x', label="Prey" if gen == 0 else "")

plt.xlabel("Strategy (1D position)")
plt.ylabel("Generation")
plt.title("Coevolution Dynamics")
plt.legend()
plt.show()